# PCA - Principal Component Analysis

Encontra as direcoes (componentes) de maxima variancia dos dados,
permitindo reduzir dimensionalidade preservando o maximo de informacao.

Implementamos via **SVD** dos dados centralizados, que e mais estavel
numericamente do que diagonalizar a matriz de covariancia.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

In [ ]:
class PCA:
    """
    Principal Component Analysis via SVD.

    1. Centraliza os dados (subtrai a media).
    2. Calcula a SVD: X = U S V^T.
    3. Os componentes principais sao as linhas de V^T.
    4. Projeta os dados nos top-k componentes.

    Usar SVD e mais estavel numericamente que diagonalizar X^T X.
    """

    def __init__(self, n_components=None, whiten=False):
        if n_components is not None and n_components <= 0:
            raise ValueError("n_components must be positive.")
        self.n_components = n_components
        self.whiten = whiten

        self.mean_ = None
        self.components_ = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X must be 2D.")
        if np.isnan(X).any():
            raise ValueError("X contains NaN values.")

        n_samples, n_features = X.shape
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        # SVD full=False e mais leve para n_samples >> n_features
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

        # garante sinal deterministico (como sklearn)
        max_abs_rows = np.argmax(np.abs(Vt), axis=1)
        signs = np.sign(Vt[np.arange(Vt.shape[0]), max_abs_rows])
        Vt *= signs[:, np.newaxis]

        explained_variance = (S ** 2) / (n_samples - 1)
        total_var = explained_variance.sum()

        k = self.n_components or n_features
        k = min(k, n_features)

        self.components_ = Vt[:k]
        self.explained_variance_ = explained_variance[:k]
        self.explained_variance_ratio_ = explained_variance[:k] / total_var
        self._singular_values_ = S[:k]
        return self

    def transform(self, X):
        if self.components_ is None:
            raise ValueError("Call fit() before transform().")
        X = np.asarray(X, dtype=float) - self.mean_
        Xt = X @ self.components_.T
        if self.whiten:
            Xt /= np.sqrt(self.explained_variance_ + 1e-12)
        return Xt

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Xt):
        if self.components_ is None:
            raise ValueError("Call fit() before inverse_transform().")
        Xt = np.asarray(Xt, dtype=float)
        if self.whiten:
            Xt = Xt * np.sqrt(self.explained_variance_ + 1e-12)
        return Xt @ self.components_ + self.mean_

In [ ]:
iris = datasets.load_iris()
X, y = iris.data, iris.target

pca = PCA(n_components=2)
Xp = pca.fit_transform(X)

print("Variancia explicada:", pca.explained_variance_ratio_)
print("Variancia cumulativa:", pca.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(7, 5))
for cls, name, color in zip([0, 1, 2], iris.target_names, ["red", "green", "blue"]):
    plt.scatter(Xp[y == cls, 0], Xp[y == cls, 1], label=name, color=color, edgecolor="k")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Iris projetado em 2 componentes principais")
plt.legend()
plt.show()

In [ ]:
# reconstrucao: quao bem os 2 componentes representam os dados originais?
X_reconstructed = pca.inverse_transform(Xp)
reconstruction_error = np.mean((X - X_reconstructed) ** 2)
print(f"Erro medio de reconstrucao: {reconstruction_error:.4f}")